In [1]:
import pandas as pd
import numpy as np
import re


from unicodedata import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering, DBSCAN

from sklearn.cluster import AffinityPropagation
from sklearn.metrics.pairwise import cosine_similarity

from difflib import SequenceMatcher


In [2]:
path = "data/train/cleaned_train.csv"

df = pd.read_csv(path)

# Clusterización de versiones


## Experimentos para un modelo especificado

In [3]:
brand = "Ford"
model = "Ecosport"

version_df = df[(df["Marca"] == brand) & (df["Modelo"] == model)].reset_index(drop=True)

### Limpieza y preprocesamiento de versiones

Este código se encarga de limpiar y normalizar los nombres de las versiones de vehículos para facilitar su posterior análisis y agrupación:

**Función** `clean_version()`
- **Normalización de caracteres**: Convierte texto a minúsculas y elimina acentos/caracteres especiales usando Unicode NFKD
- **Filtrado de caracteres**: Solo mantiene letras, números y espacios, eliminando símbolos y puntuación
- **Limpieza de espacios**: Normaliza espacios múltiples a espacios únicos y elimina espacios al inicio/final
- **Manejo de valores nulos**: Convierte valores `NaN` a strings vacíos


In [4]:
def clean_version(s: str) -> str:
    s = "" if pd.isna(s) else s
    s = normalize("NFKD", s.lower()).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

version_df["version_clean"] = version_df["Versión"].apply(clean_version)
version_df = version_df[version_df["version_clean"] != ""].reset_index(drop=True)

print("Número de versiones únicas:", version_df["version_clean"].nunique())

print("\nPrimeras 5 versiones únicas:")
print(version_df["version_clean"].value_counts()[:5])

Número de versiones únicas: 49

Primeras 5 versiones únicas:
version_clean
se           375
titanium     259
freestyle    144
s             92
my10 xls      46
Name: count, dtype: int64


### HAC y TF-IDF

In [5]:
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"])

non_zero_mask = np.array(X.sum(axis=1)).flatten() > 0
X_filtered = X[non_zero_mask].toarray()
taos_df_filtered = version_df[non_zero_mask].copy()

agg = AgglomerativeClustering(
    metric="cosine",
    linkage="average",
    distance_threshold=0.8,
    n_clusters=None,
)

# Fit on filtered data
clusters_filtered = agg.fit_predict(X_filtered)

# Assign clusters back to original dataframe
version_df["cluster"] = -1  # Default for zero vectors
version_df.loc[non_zero_mask, "cluster"] = clusters_filtered

In [6]:
canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary

,cluster,version_canon,count
0,-1,se,467
1,0,titanium,323
3,2,freestyle,178
2,1,my10 xls,140
13,12,xls,42
5,4,xls mp3,28
7,6,se tdci,20
8,7,gdi storm,12
10,9,d se,9
4,3,se l18,6


### Affinity propagation

In [7]:
S = cosine_similarity(X)

pref = np.median(S) - 0.05
ap = AffinityPropagation(affinity="precomputed",
                         damping=0.95,
                         preference=pref,
                         max_iter=500,
                         convergence_iter=16)

version_df["cluster"] = ap.fit_predict(S)


canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
summary


,cluster,version_canon,count
3,3,titanium,348
0,0,se,323
2,2,se,316
1,1,se,250


### DBSCAN

- Leer `CountVectorizer` en [Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) (podemos jugar con los argumentos)

In [8]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"]).toarray()

print(X.shape)

(1237, 377)


In [9]:
n_versions = version_df["version_clean"].nunique()

db = DBSCAN(eps=0.3, min_samples=max(2, n_versions // 4), metric="cosine")
version_df["cluster"] = db.fit_predict(X)

# canónico solo para clusters válidos (≠ –1)
canon = (
    version_df[version_df["cluster"] != -1]
      .groupby("cluster")["Versión"]
      .agg(lambda s: s.value_counts().idxmax())
)
version_df["version_canon"] = version_df["cluster"].map(canon)

# todo el ruido recibe una misma etiqueta
version_df.loc[version_df["cluster"] == -1, "version_canon"] = "Unassigned"

summary = (
    version_df.groupby(["cluster", "version_canon"])
              .size()
              .reset_index(name="count")
              .sort_values("count", ascending=False)
)
print(summary)
print("\nClusters densos:", canon.size,
      "  Etiquetas finales:", version_df["version_canon"].nunique())


   cluster     version_canon  count
0       -1        Unassigned    521
1        0          titanium    321
3        2         freestyle    178
5        4  my10 xl plus mp3     60
7        6          my10 xls     59
6        5               xls     42
4        3          xlt plus     21
8        7           se tdci     18
2        1           xls mp3     17

Clusters densos: 8   Etiquetas finales: 9


### Usar los datos de `mercadolibre_versions.csv`

In [10]:
ml_versions = pd.read_csv('data/mercadolibre_versions.csv')
ford_ml = ml_versions[(ml_versions['Brand'] == brand) &
                      (ml_versions['Model'] == model ) &
                      (ml_versions['Version'] != 'N/A')]

real_versions = ford_ml['Version'].tolist()

clustered_counts = version_df['Versión'].value_counts()
print(clustered_counts.head())

threshold = 0.3
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3), lowercase=True)

final_mapping = {}
mapping_results = []

for clustered in clustered_counts.index:

    best_match = None
    best_score = 0.0
    clean_clustered = clean_version(clustered)


    for rv in real_versions:
        clean_real = clean_version(rv)
        if clean_clustered and clean_real:
            X = vectorizer.fit_transform([clean_clustered, clean_real])
            score = cosine_similarity(X[0:1], X[1:2])[0][0]
        else:
            score = 0.0
        if score > best_score:
            best_score = score
            best_match = rv
    if best_score < threshold:
        best_match = 'Se'
        
    final_mapping[clustered] = best_match
    mapping_results.append({
        'clustered_version': clustered,
        'count': clustered_counts[clustered],
        'mapped_to': best_match,
        'similarity_score': best_score
    })

mapping_df = pd.DataFrame(mapping_results)
print(mapping_df.to_string(index=False))

Versión
se           375
titanium     259
freestyle    144
s             92
my10 xls      46
Name: count, dtype: int64
       clustered_version  count mapped_to  similarity_score
                      se    375        Se          1.000000
                titanium    259  Titanium          1.000000
               freestyle    144 Freestyle          1.000000
                       s     92         S          1.000000
                my10 xls     46  My10 Xls          1.000000
                     xls     42       Xls          1.000000
        my10 xl plus mp3     33  My10 Xls          0.664784
            gdi titanium     28  Titanium          0.843338
           freestyle 143     22 Freestyle          0.845154
titanium powershift dps6     18  Titanium          0.621825
                 se tdci     18        Se          0.408248
             xl plus mp3     18   Xls Mp3          0.707107
                 xls mp3     17   Xls Mp3          1.000000
           my10 xls plus     12  My10 Xls

In [11]:
print("Before mapping:")
print("Original version counts:", version_df["Versión"].value_counts())
print(f"Total unique versions: {version_df['Versión'].nunique()}")

version_df["Versión"] = version_df["Versión"].map(final_mapping)

print("\nAfter mapping:")
print("Mapped version counts:", version_df["Versión"].value_counts())
print(f"Total unique versions: {version_df['Versión'].nunique()}")

# Verify that all versions are now real MercadoLibre versions
mapped_versions = set(version_df["Versión"].unique())
real_versions_set = set(real_versions)

print(f"\nValidation:")
print(f"All mapped versions are real versions: {mapped_versions.issubset(real_versions_set)}")
print(f"Mapped versions: {sorted(mapped_versions)}")
print(f"Real versions not used: {sorted(real_versions_set - mapped_versions)}")

# Show final summary
final_summary = version_df["Versión"].value_counts().reset_index()
final_summary.columns = ["Version", "Count"]
final_summary["Percentage"] = (final_summary["Count"] / final_summary["Count"].sum() * 100).round(2)

print(f"\nFinal Version Distribution:")
print("="*50)
print(final_summary.to_string(index=False))

Before mapping:
Original version counts: Versión
se                          375
titanium                    259
freestyle                   144
s                            92
my10 xls                     46
xls                          42
my10 xl plus mp3             33
gdi titanium                 28
freestyle 143                22
titanium powershift dps6     18
se tdci                      18
xl plus mp3                  18
xls mp3                      17
my10 xls plus                12
xlt plus                     11
my10 xlt plus                10
d se                          9
xl plus                       8
gdi storm                     7
gdi freestyle                 6
titanium powershift           6
tdci xls mp3                  5
xlt                           5
tdci xls                      4
gdi se                        4
se l18                        4
freestyle l18                 3
titanium l18                  3
gdi                           3
s tdci                 

## Asignar cada muestra al cluster que pertenece

In [12]:
version_df['Versión'] = version_df['Versión'].map(final_mapping)

summary = version_df['Versión'].value_counts().reset_index()
summary.columns = ['Version', 'Count']
summary['Percentage'] = (summary['Count'] / summary['Count'].sum() * 100).round(2)

print(summary.to_string(index=False))


Empty DataFrame
Columns: [Version, Count, Percentage]
Index: []


# Compartir features entre cada versión

Ejemplo: Si un auto en particular de una cierta versión tiene turbo, y los demás autos de esa versión no lo especifican, entonces se le asigna turbo a los demás autos de esa versión.

In [13]:
def fill_by_mode(df, col, keys=("Marca", "Modelo", "Versión")):
    mode_vals = (
        df.groupby(list(keys))[col]
          .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    )
    df[col] = df[col].fillna(mode_vals)

cols_to_fill = ["cv", "Motor", "Tracción", "Turbo"]

for c in cols_to_fill:
    if c in version_df.columns:
        fill_by_mode(version_df, c)

print("Faltantes después de imputar:")
print(version_df[cols_to_fill].isna().sum())

Faltantes después de imputar:
cv          319
Motor         1
Tracción     99
Turbo         0
dtype: int64


# Color

In [14]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\valen\Documents\Repositorios\MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# Análisis inicial de colores
color_counts = df['Color'].value_counts()
print("Cantidad de colores únicos:", len(color_counts), "\n")

with pd.option_context('display.max_rows', None):
    print(color_counts)

Cantidad de colores únicos: 64 

Color
Gris                             4395
Blanco                           3817
Negro                            2264
Plateado                         1132
Rojo                              904
Azul                              887
Marrón                            213
Dorado                            150
Verde                             107
Beige                             100
Celeste                            74
Naranja                            58
Gris oscuro                        35
Bordó                              28
Violeta                            24
Amarillo                           21
Rosa                                7
Plata                               5
Otro                                5
Acero                               3
Negra                               3
Blanca                              3
Gris plata                          3
Marrón oscuro                       2
MINERALWEISS METALLIC               2
Café       

In [16]:
# Función para limpiar y normalizar colores
def clean_color(color):
 
    color_str = str(color).strip().lower()    
    
    return color_str

df['Color_clean'] = df['Color'].apply(clean_color).dropna()


# Verificar colores únicos después de la limpieza
color_counts_clean = df['Color_clean'].value_counts()
print("Cantidad de colores únicos después de limpieza:", len(color_counts_clean), "\n")
print("Top 15 colores más frecuentes:")
print(color_counts_clean.head(15))

Cantidad de colores únicos después de limpieza: 60 

Top 15 colores más frecuentes:
Color_clean
gris           4395
blanco         3817
negro          2264
plateado       1132
rojo            904
azul            887
nan             315
marrón          213
dorado          150
verde           107
beige           100
celeste          74
naranja          58
gris oscuro      35
bordó            28
Name: count, dtype: int64


In [17]:
# Definir el número N de colores más frecuentes a mantener
N = 16

# Obtener los N colores más frecuentes
top_colors = color_counts_clean.head(N).index.tolist()
print(f"Los {N} colores más frecuentes que se mantendrán:")
for i, color in enumerate(top_colors, 1):
    print(f"{i}. {color} ({color_counts_clean[color]} casos)")

Los 16 colores más frecuentes que se mantendrán:
1. gris (4395 casos)
2. blanco (3817 casos)
3. negro (2264 casos)
4. plateado (1132 casos)
5. rojo (904 casos)
6. azul (887 casos)
7. nan (315 casos)
8. marrón (213 casos)
9. dorado (150 casos)
10. verde (107 casos)
11. beige (100 casos)
12. celeste (74 casos)
13. naranja (58 casos)
14. gris oscuro (35 casos)
15. bordó (28 casos)
16. violeta (24 casos)


### Entrenar modelo de Word2Vec

In [18]:
# Cargar modelo preentrenado multilingüe
print("Cargando modelo de sentence transformers...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Modelo cargado exitosamente!")

# Obtener todos los colores únicos
all_colors = [color for color in df['Color_clean'].unique() if color and color not in ['', 'nan']]

print(f"Generando embeddings para {len(all_colors)} colores únicos...")

# Generar embeddings para todos los colores
color_embeddings = model.encode(all_colors)

print(f"Embeddings generados. Dimensión: {color_embeddings.shape}")

# Crear diccionario para mapear colores a sus embeddings
color_to_embedding = {color: embedding for color, embedding in zip(all_colors, color_embeddings)}

Cargando modelo de sentence transformers...
Modelo cargado exitosamente!
Generando embeddings para 59 colores únicos...
Embeddings generados. Dimensión: (59, 384)


In [19]:
# Arreglar la función de similitud
def find_most_similar_color(target_color, top_colors, color_to_embedding):
    """Encuentra el color más similar entre los colores top usando embeddings preentrenados"""
    if target_color in top_colors:
        return target_color  # Si ya está en los top, no cambiar
    
    if target_color not in color_to_embedding:
        # Si no se encuentra el color, asignar al más frecuente
        return top_colors[0]
    
    # Obtener embedding del color objetivo
    target_embedding = color_to_embedding[target_color].reshape(1, -1)
    
    # Calcular similitudes con todos los colores top
    similarities = {}
    for top_color in top_colors:
        if top_color in color_to_embedding:
            top_embedding = color_to_embedding[top_color].reshape(1, -1)
            similarity = cosine_similarity(target_embedding, top_embedding)[0][0]
            similarities[top_color] = similarity
    
    # Debug: mostrar similitudes para algunos colores
    if target_color in ['blanca', 'negra', 'plata', 'café']:
        print(f"\nSimilitudes para '{target_color}':")
        for top_color, sim in sorted(similarities.items(), key=lambda x: x[1], reverse=True):
            print(f"  {top_color}: {sim:.4f}")
    
    # Retornar el color con mayor similitud
    if similarities:
        best_color = max(similarities, key=similarities.get)
        return best_color
    else:
        return top_colors[0]  # Fallback al más frecuente

# Aplicar la clusterización corregida
print("Aplicando clusterización de colores (versión corregida)...")

# Crear diccionario de mapeo para colores no top
color_mapping = {}
colors_to_process = [color for color in df['Color_clean'].unique() 
                    if color and color not in ['', 'nan'] and color not in top_colors]

print(f"Procesando {len(colors_to_process)} colores únicos que no están en los top {N}...")

for color in colors_to_process:
    most_similar = find_most_similar_color_fixed(color, top_colors, color_to_embedding)
    color_mapping[color] = most_similar
    print(f"  '{color}' -> '{most_similar}'")

print("\nMapeo completado!")

Aplicando clusterización de colores (versión corregida)...
Procesando 44 colores únicos que no están en los top 16...

Similitudes para 'blanca':
  blanco: 0.9784
  beige: 0.6826
  bordó: 0.6663
  dorado: 0.6512
  plateado: 0.6287
  negro: 0.6222
  marrón: 0.6206
  gris: 0.5755
  verde: 0.5369
  gris oscuro: 0.5072
  violeta: 0.4752
  azul: 0.4654
  rojo: 0.4653
  celeste: 0.4280
  naranja: 0.4265
  'blanca' -> 'blanco'
  'marrón oscuro' -> 'gris oscuro'
  'rosa' -> 'beige'
  'gris titane' -> 'gris'
  'gris plata' -> 'gris oscuro'
  'grafito' -> 'negro'
  'amarillo' -> 'beige'
  'verde oscuro' -> 'verde'
  'blanco glaciar' -> 'blanco'

Similitudes para 'negra':
  negro: 0.9792
  gris oscuro: 0.7801
  beige: 0.7107
  gris: 0.6867
  plateado: 0.6577
  marrón: 0.6535
  bordó: 0.6317
  dorado: 0.6112
  blanco: 0.5802
  verde: 0.5530
  violeta: 0.5274
  azul: 0.5152
  celeste: 0.4880
  rojo: 0.4562
  naranja: 0.3453
  'negra' -> 'negro'
  'carbon black' -> 'negro'
  'skyscraper grau metalli

### Aplicar clustering de colores

In [ ]:
print("Aplicando clusterización de colores...")

# Crear diccionario de mapeo para colores no top
color_mapping = {}
colors_to_process = [color for color in df['Color_clean'].unique() 
                    if color and color not in ['', 'nan'] and color not in top_colors]

print(f"Procesando {len(colors_to_process)} colores únicos que no están en los top {N}...")

for color in colors_to_process:
    most_similar = find_most_similar_color(color, top_colors, model)
    color_mapping[color] = most_similar
    print(f"  '{color}' -> '{most_similar}'")

print("\nMapeo completado!")

Aplicando clusterización de colores...
Procesando 44 colores únicos que no están en los top 16...


NameError: name 'find_most_similar_color' is not defined

In [76]:
# Mostrar similitudes detalladas para algunos ejemplos
print("Similitudes detalladas para algunos colores de ejemplo:")

example_colors = colors_to_process[:10]  # Tomar los primeros 10 como ejemplo

for target_color in example_colors:
    if target_color in color_to_embedding:
        target_embedding = color_to_embedding[target_color].reshape(1, -1)
        
        print(f"\nColor: '{target_color}'")
        print("Similitudes con colores top:")
        
        similarities = []
        for top_color in top_colors:
            if top_color in color_to_embedding:
                top_embedding = color_to_embedding[top_color].reshape(1, -1)
                similarity = cosine_similarity(target_embedding, top_embedding)[0][0]
                similarities.append((top_color, similarity))
        
        # Ordenar por similitud descendente
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        for top_color, sim in similarities[:3]:  # Mostrar solo los top 3
            print(f"  {top_color}: {sim:.4f}")
        
        print(f"  -> Asignado a: '{similarities[0][0]}'")

Similitudes detalladas para algunos colores de ejemplo:

Color: 'blanca'
Similitudes con colores top:
  blanco: 0.9784
  beige: 0.6826
  bordó: 0.6663
  -> Asignado a: 'blanco'

Color: 'marrón oscuro'
Similitudes con colores top:
  gris oscuro: 0.8628
  negro: 0.8271
  beige: 0.7595
  -> Asignado a: 'gris oscuro'

Color: 'rosa'
Similitudes con colores top:
  beige: 0.6801
  violeta: 0.6315
  dorado: 0.5878
  -> Asignado a: 'beige'

Color: 'gris titane'
Similitudes con colores top:
  gris: 0.6409
  negro: 0.5502
  gris oscuro: 0.5460
  -> Asignado a: 'gris'

Color: 'gris plata'
Similitudes con colores top:
  gris oscuro: 0.7622
  negro: 0.7335
  gris: 0.6620
  -> Asignado a: 'gris oscuro'

Color: 'grafito'
Similitudes con colores top:
  negro: 0.5662
  beige: 0.5451
  bordó: 0.4846
  -> Asignado a: 'negro'

Color: 'amarillo'
Similitudes con colores top:
  beige: 0.6049
  naranja: 0.5944
  dorado: 0.5292
  -> Asignado a: 'beige'

Color: 'verde oscuro'
Similitudes con colores top:
  verde

In [79]:
# Debuggear el problema de similitudes
print("=== DEBUGGING SIMILITUDES ===")

# Verificar que los embeddings se generaron correctamente
print(f"Shape de embeddings: {color_embeddings.shape}")
print(f"Rango de valores: {color_embeddings.min():.4f} a {color_embeddings.max():.4f}")

# Verificar embeddings de colores top
print("\nEmbeddings de colores top:")
for color in top_colors[:3]:
    if color in color_to_embedding:
        emb = color_to_embedding[color]
        print(f"  {color}: shape {emb.shape}, norm {np.linalg.norm(emb):.4f}")

# Verificar similitudes entre colores top
print("\nSimilitudes entre colores top:")
for i, color1 in enumerate(top_colors[:3]):
    for j, color2 in enumerate(top_colors[:3]):
        if i != j and color1 in color_to_embedding and color2 in color_to_embedding:
            emb1 = color_to_embedding[color1].reshape(1, -1)
            emb2 = color_to_embedding[color2].reshape(1, -1)
            sim = cosine_similarity(emb1, emb2)[0][0]
            print(f"  {color1} vs {color2}: {sim:.4f}")

=== DEBUGGING SIMILITUDES ===
Shape de embeddings: (59, 384)
Rango de valores: -0.9980 a 2.1870

Embeddings de colores top:
  gris: shape (384,), norm 3.7113
  blanco: shape (384,), norm 4.5200
  negro: shape (384,), norm 4.4994

Similitudes entre colores top:
  gris vs blanco: 0.5995
  gris vs negro: 0.7442
  blanco vs gris: 0.5995
  blanco vs negro: 0.6373
  negro vs gris: 0.7442
  negro vs blanco: 0.6373
